# VisionStream AI — Sprint 3: Remote NVIDIA GPU Lab
## 02. Canonical YOLOv8n CUDA Inference & Baseline Benchmarking

**Tujuan:** Menjalankan inferensi model deteksi kanonikal YOLOv8n pada GPU NVIDIA menggunakan Ultralytics PyTorch backend (FP32 & FP16 AMP) dan mengukur latensi serta throughput.

> **Catatan Tata Kelola:** Model yang digunakan adalah canonical baseline `yolov8n.pt` (SHA-256: `F59B3D833E2FF32E194B5BB8E08D211DC7C5BDF144B90D2C8412C47CCFC83B36`).

### Step 1: Install Notebook-Isolated Dependencies

In [ ]:
!pip install --quiet ultralytics opencv-python-headless

### Step 2: Acquire Canonical YOLOv8n Baseline & Verify Integrity
*Cell ini mengunduh atau memuat `yolov8n.pt` resmi dan memverifikasi SHA-256 kanonikal.*

In [ ]:
import hashlib, urllib.request, os

MODEL_PATH = "yolov8n.pt"
EXPECTED_HASH = "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36"

if not os.path.exists(MODEL_PATH):
    print("Acquiring official YOLOv8n release weight...")
    url = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt"
    urllib.request.urlretrieve(url, MODEL_PATH)

with open(MODEL_PATH, "rb") as f:
    h = hashlib.sha256(f.read()).hexdigest().lower()

print(f"Model File: {MODEL_PATH}")
print(f"SHA-256:    {h}")
print(f"Expected:   {EXPECTED_HASH}")
assert h == EXPECTED_HASH, f"Hash mismatch! Got {h}, expected {EXPECTED_HASH}"
print("Integrity verification: PASS")

### Step 3: Run CUDA Inference Benchmark (Synthetic & Warmup)
*Menggunakan dummy tensor 640x640 sesuai protokol kanonikal VisionStream AI.*

In [ ]:
import time, json
import numpy as np
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), "NVIDIA CUDA is required!"

model = YOLO(MODEL_PATH)
device = "cuda:0"

# Create synthetic batch (640x640x3 BGR)
dummy_frame = np.random.randint(0, 256, (640, 640, 3), dtype=np.uint8)

WARMUP_RUNS = 20
MEASURED_RUNS = 200

print(f"Running {WARMUP_RUNS} warm-up iterations on {device}...")
for _ in range(WARMUP_RUNS):
    _ = model.predict(source=dummy_frame, device=device, conf=0.35, iou=0.65, classes=[0], verbose=False)
torch.cuda.synchronize()
print("Warm-up complete.")

print(f"Running {MEASURED_RUNS} benchmark iterations on {device}...")
latencies_ms = []
for _ in range(MEASURED_RUNS):
    t0 = time.perf_counter()
    res = model.predict(source=dummy_frame, device=device, conf=0.35, iou=0.65, classes=[0], verbose=False)
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    latencies_ms.append((t1 - t0) * 1000.0)

mean_lat = float(np.mean(latencies_ms))
med_lat = float(np.median(latencies_ms))
p95_lat = float(np.percentile(latencies_ms, 95))
fps = float(1000.0 / mean_lat)

print(f"Mean Inference Latency:   {mean_lat:.2f} ms")
print(f"Median Inference Latency: {med_lat:.2f} ms")
print(f"P95 Inference Latency:    {p95_lat:.2f} ms")
print(f"Throughput:               {fps:.2f} FPS")

results_data = {
    "device": torch.cuda.get_device_name(0),
    "model": MODEL_PATH,
    "warmup_runs": WARMUP_RUNS,
    "measured_runs": MEASURED_RUNS,
    "mean_latency_ms": mean_lat,
    "median_latency_ms": med_lat,
    "p95_latency_ms": p95_lat,
    "fps": fps
}

with open("cuda_inference_benchmark.json", "w") as f:
    json.dump(results_data, f, indent=2)
print("Benchmark results exported to cuda_inference_benchmark.json")